# 0차시 — 설치 확인 (Installation Check)

이 노트북을 **위에서부터 실행**(`Shift+Enter`)하면, 워크샵 실습에 필요한 도구가 모두 설치됐는지 표로 알려줍니다.

- ✅ = 준비됨 / ❌ = 빠짐(또는 버전 부족) 
- 이 노트북 자체가 열려 실행된다면 **Python·Jupyter·VS Code** 는 이미 OK 입니다.


## 1) 도구 설치 점검
아래 셀을 실행하세요.

In [4]:
import shutil, subprocess, os, sys, re

def ver(cmd):
    """명령을 실행해 버전 문자열을 반환. 없으면 None."""
    exe = cmd[0]
    if shutil.which(exe) is None:
        return None
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=20)
        return (out.stdout + out.stderr).strip().splitlines()[0]
    except Exception as e:
        return f"(확인 실패: {e})"

def major(s):
    """버전 문자열에서 첫 숫자(major)를 추출."""
    m = re.search(r'(\d+)', s or "")
    return int(m.group(1)) if m else None

checks = [
    ("Git",        ["git", "--version"],     None),
    ("Node.js",    ["node", "--version"],    22),   # 22 이상 권장
    ("npm",        ["npm", "--version"],     None),
    ("Python",     [sys.executable, "--version"], 3),
    ("uv",         ["uv", "--version"],      None),
    ("Codex CLI",  ["codex", "--version"],   None),
    ("VS Code",    ["code", "--version"],    None),  # PATH 미등록이면 ❌여도 무방
]

print(f"{'도구':<12}{'상태':<6}{'버전/메모'}")
print("-" * 50)
missing = []
for name, cmd, min_major in checks:
    v = ver(cmd)
    if v is None:
        status, note = "❌", "설치 안 됨 (또는 PATH 미등록)"
        missing.append(name)
    elif min_major and (major(v) or 0) < min_major:
        status, note = "⚠️", f"{v}  → {min_major}+ 권장"
        missing.append(name + "(버전)")
    else:
        status, note = "✅", v
    print(f"{name:<12}{status:<5}{note}")

# OpenAI API 키
print("-" * 50)
key = os.environ.get("OPENAI_API_KEY", "")
if key.startswith("sk-"):
    print(f"OpenAI 키   ✅  환경변수 OPENAI_API_KEY 등록됨 (sk-...{key[-4:]})")
else:
    print("OpenAI 키   ➖  환경변수 미등록 — 각 실습 노트북 첫 셀에서 직접 입력하면 됩니다(정상)")

# openai 패키지
print("-" * 50)
try:
    import openai
    print(f"openai 패키지 ✅  {openai.__version__}")
except ImportError:
    print("openai 패키지 ➖  미설치 — 각 실습 노트북 첫 셀의 `%pip install -q openai` 가 자동 설치합니다")

# Docker — 3일차(17·18차시) Langfuse 자체 호스팅용 (1~6차시엔 불필요 → missing에 넣지 않음)
print("-" * 50)
dv = ver(["docker", "--version"])
if dv:
    print(f"Docker      ✅  {dv}  (3일차 Langfuse 자체 호스팅용)")
else:
    print("Docker      ➖  미설치 — 3일차(17·18차시) Langfuse 자체 호스팅용입니다.")
    print("              지금은 정상! 3일차 전까지 설치하면 됩니다(설치_가이드.md의 Docker 항목 참고).")

print("-" * 50)
if missing:
    print("⚠️ 빠진 항목:", ", ".join(missing))
    print("→ 해당 항목을 설치한 뒤, 터미널/VS Code를 새로 열고 다시 실행하세요.")
else:
    print("🎉 핵심 도구가 모두 준비됐습니다! 1차시로 진행하세요. (Docker는 3일차 전까지 준비하면 됩니다)")


도구          상태    버전/메모
--------------------------------------------------
Git         ✅    git version 2.43.0.windows.1
Node.js     ⚠️   v20.19.0  → 22+ 권장
npm         ✅    (확인 실패: [WinError 2] 지정된 파일을 찾을 수 없습니다)
Python      ✅    Python 3.11.9
uv          ✅    uv 0.8.3 (7e78f54e7 2025-07-24)
Codex CLI   ✅    (확인 실패: [WinError 2] 지정된 파일을 찾을 수 없습니다)
VS Code     ✅    (확인 실패: [WinError 2] 지정된 파일을 찾을 수 없습니다)
--------------------------------------------------
OpenAI 키   ✅  환경변수 OPENAI_API_KEY 등록됨 (sk-...XzQA)
--------------------------------------------------
openai 패키지 ✅  2.45.0
--------------------------------------------------
Docker      ✅  Docker version 27.2.0, build 3ab4256  (3일차 Langfuse 자체 호스팅용)
--------------------------------------------------
⚠️ 빠진 항목: Node.js(버전)
→ 해당 항목을 설치한 뒤, 터미널/VS Code를 새로 열고 다시 실행하세요.


## 2) (선택) OpenAI API 키 실제 동작 확인
키와 크레딧이 준비됐는지 1회 호출로 확인합니다. 비용은 매우 작습니다(약 $0.0001).

In [3]:
# (선택) OpenAI 키가 실제로 동작하는지 1회 호출로 확인
# 키가 없으면 입력창이 뜹니다(화면에 안 보임). 크레딧이 없으면 인증/한도 오류가 날 수 있습니다.
import os, getpass
try:
    import openai
except ImportError:
    %pip install -q openai
    import openai

from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY 붙여넣기: ")

try:
    client = OpenAI()
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "워크샵 준비 완료를 한 문장으로 축하해줘(한국어)."}],
    )
    print("✅ API 호출 성공:", r.choices[0].message.content)
except Exception as e:
    print("❌ API 호출 실패:", e)
    print("→ 키 오타/공백, platform.openai.com 결제수단·크레딧 등록 여부를 확인하세요.")


✅ API 호출 성공: 워크샵 준비를 완벽하게 마치신 것을 진심으로 축하드립니다!
